In [1]:
from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path
from collections import Counter

DATA_DIR = Path("/content/drive/MyDrive/greek-news-nlp")
EXPECTED = {"train": 3675, "val": 787, "test": 788}

# Έλεγχος ότι υπάρχουν όλα τα αρχεία.
paths = {
    split: DATA_DIR / f"el_{split}.jsonl"
    for split in EXPECTED
}

missing = [p.name for p in paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

# Φόρτωση και έλεγχος του corpus.
docs = []

for split, path in paths.items():
    with path.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    print(f"{path.name}: {len(rows)} articles")
    assert len(rows) == EXPECTED[split], f"Unexpected size: {split}"

    for row in rows:
        docs.append({
            "id": row["document_id"],
            "text": row["text_clean"],
            "label": row["GPT-IPTC-label"],
            "split": split,
        })

counts = Counter(doc["id"] for doc in docs)
duplicates = [doc_id for doc_id, count in counts.items() if count > 1]
empty = [
    doc["id"] for doc in docs
    if not isinstance(doc["text"], str) or not doc["text"].strip()
]

print("\nTotal articles:", len(docs))
print("Unique IDs:", len(counts))
print("Duplicate IDs:", len(duplicates))
print("Empty texts:", len(empty))

assert not duplicates, "Duplicate document IDs found"
assert not empty, "Empty article texts found"

print("\nOK — corpus ready")

Mounted at /content/drive
el_train.jsonl: 3675 articles
el_val.jsonl: 787 articles
el_test.jsonl: 788 articles

Total articles: 5250
Unique IDs: 5250
Duplicate IDs: 0
Empty texts: 0

OK — corpus ready


In [2]:
%pip install -q rank-bm25==0.2.2

In [3]:
import re
import unicodedata

import numpy as np
from rank_bm25 import BM25Okapi

def tokenize_greek(text):
    # Πεζά, ενοποίηση σ/ς και αφαίρεση τόνων.
    text = unicodedata.normalize("NFD", text.casefold())
    text = "".join(
        char for char in text
        if unicodedata.category(char) != "Mn"
    )
    return re.findall(r"[^\W_]+", text, flags=re.UNICODE)

tokenized_docs = [tokenize_greek(doc["text"]) for doc in docs]

assert all(tokenized_docs), "Some articles have no searchable tokens"

bm25_full = BM25Okapi(
    tokenized_docs,
    k1=1.5,
    b=0.75,
    epsilon=0.25,
)

print("BM25 indexed articles:", bm25_full.corpus_size)

def search_bm25(query, k=5):
    tokens = tokenize_greek(query)
    if not tokens or k <= 0:
        return []

    scores = bm25_full.get_scores(tokens)

    # Σταθερή σειρά σε περίπτωση ισοβαθμίας.
    order = np.argsort(-scores, kind="stable")
    results = []

    for i in order:
        if scores[i] <= 0:
            continue

        results.append({
            "id": docs[i]["id"],
            "score": float(scores[i]),
            "label": docs[i]["label"],
            "snippet": docs[i]["text"][:350],
        })

        if len(results) == k:
            break

    return results

query = "πλημμύρες και αποζημιώσεις αγροτών"

for rank, result in enumerate(search_bm25(query), start=1):
    print(f"\n#{rank} | {result['id']} | score={result['score']:.3f}")
    print("Category:", result["label"])
    print(result["snippet"])

BM25 indexed articles: 5250

#1 | macocu.el.9211668 | score=15.905
Category: disaster, accident and emergency incident
Πειθαρχικές ευθύνες επιρρίπτει το πόρισμα για τις πλημμύρες στη Μάνδρα Σε σειρά διαφορετικών φορέων του δημοσίου εντοπίζει ευθύνες για τις φονικές πλημμύρες του 2017 στη Μάνδρα Αττικής η Γενική Επιθεωρήτρια Δημόσιας Διοίκησης, Μαρία Παπασπύρου, στο πόρισμά της. Το δασαρχείο και την πολεοδομία Ελευσίνας, εκλεγμένους του Δήμου Μάνδρας και ιδιώτες μηχ

#2 | macocu.el.712626 | score=13.941
Category: disaster, accident and emergency incident
Η Νομαρχιακή Επιτροπή Τουριστικής Προβολής της Ν.Α. Ευβοίας έχοντας ως στόχο την προβολή της Εύβοιας στις ξένες τουριστικές αγορές φιλοξένησε μετά τους Ολλανδούς, Βέλγους, Ούγγρους, Ρουμάνους δημοσιογράφους και Ρώσους τουριστικούς πράκτορες ένα συνεργείο 4 Ρουμάνων δημοσιογράφων του τηλεοπτικού σταθμού EYRO CHANNEL (26-31/8/09) και δέκα Γερμανούς δ

#3 | macocu.el.14374889 | score=13.656
Category: disaster, accident and emergency incide

In [4]:
# Αναζήτηση του αποθηκευμένου dense index στον φάκελο του project.
index_candidates = sorted(DATA_DIR.rglob("articles.faiss"))

if not index_candidates:
    print("Δεν βρέθηκε articles.faiss στον φάκελο:", DATA_DIR)
    print("Θα χρειαστεί να φορτώσουμε το index από το Hugging Face.")
else:
    for index_path in index_candidates:
        docs_path = index_path.parent / "docs.json"

        print("\nIndex:", index_path)
        print("docs.json exists:", docs_path.is_file())

        if docs_path.is_file():
            with docs_path.open(encoding="utf-8") as f:
                indexed_docs = json.load(f)

            current_by_id = {doc["id"]: doc["text"] for doc in docs}
            indexed_ids = [doc["id"] for doc in indexed_docs]

            same_ids = (
                len(indexed_ids) == len(current_by_id)
                and set(indexed_ids) == set(current_by_id)
            )
            same_texts = same_ids and all(
                doc["text"] == current_by_id[doc["id"]]
                for doc in indexed_docs
            )

            print("Indexed articles:", len(indexed_docs))
            print("Same document IDs:", same_ids)
            print("Same article texts:", same_texts)
            print(
                "Same document order:",
                indexed_ids == [doc["id"] for doc in docs],
            )

Δεν βρέθηκε articles.faiss στον φάκελο: /content/drive/MyDrive/greek-news-nlp
Θα χρειαστεί να φορτώσουμε το index από το Hugging Face.


In [5]:
%pip install -q huggingface-hub faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.2 MB/s eta 0:00:00


In [6]:
import faiss
from huggingface_hub import HfApi, hf_hub_download

INDEX_REPO = "Tilemachos-P/greek-news-index"
INDEX_DIR = DATA_DIR / "rag_index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# Καταγραφή της έκδοσης που κατεβάζουμε.
index_revision = HfApi().repo_info(
    repo_id=INDEX_REPO,
    repo_type="dataset",
).sha

downloaded = {}

for filename in ("articles.faiss", "docs.json"):
    downloaded[filename] = hf_hub_download(
        repo_id=INDEX_REPO,
        repo_type="dataset",
        revision=index_revision,
        filename=filename,
        local_dir=str(INDEX_DIR),
    )

with open(downloaded["docs.json"], encoding="utf-8") as f:
    dense_docs = json.load(f)

dense_index = faiss.read_index(downloaded["articles.faiss"])

current_by_id = {doc["id"]: doc["text"] for doc in docs}
dense_ids = [doc["id"] for doc in dense_docs]

assert len(dense_ids) == len(set(dense_ids)), "Duplicate IDs in index"
assert set(dense_ids) == set(current_by_id), "Document IDs differ"
assert all(
    doc["text"] == current_by_id[doc["id"]]
    for doc in dense_docs
), "Article texts differ"
assert dense_index.ntotal == len(dense_docs), "Index size mismatch"

# Αντιστοίχιση μέσω ID, ακόμη και αν η σειρά είναι διαφορετική.
dense_row_to_id = dense_ids
doc_by_id = {doc["id"]: doc for doc in docs}

print("Repository revision:", index_revision)
print("FAISS vectors:", dense_index.ntotal)
print("Embedding dimensions:", dense_index.d)
print("Same document order:", dense_ids == [doc["id"] for doc in docs])
print("Saved to:", INDEX_DIR)
print("\nOK — dense index loaded and corpus verified")

articles.faiss: reconstructing file:   0%|          |  0.00B / 8.06MB            

articles.faiss: downloading bytes:           |  0.00B            

docs.json: reconstructing file:   0%|          |  0.00B / 15.3MB            

docs.json: downloading bytes:           |  0.00B            

Repository revision: 17c079f7c9c6a71a8a1747d8a219594260096d0a
FAISS vectors: 5250
Embedding dimensions: 384
Same document order: True
Saved to: /content/drive/MyDrive/greek-news-nlp/rag_index

OK — dense index loaded and corpus verified


In [7]:
%pip install -q sentence-transformers

In [8]:
import torch
import importlib.metadata as metadata
from sentence_transformers import SentenceTransformer

# 1. Load embedding model explicitly on CPU.
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cpu",
)

embedder.max_seq_length = 256
embedder.eval()

# 2. Verify compatibility with existing FAISS index.
embedding_dim = embedder.get_sentence_embedding_dimension()

assert embedding_dim == dense_index.d, (
    f"Dimension mismatch: model={embedding_dim}, "
    f"FAISS={dense_index.d}"
)

assert dense_index.metric_type == faiss.METRIC_INNER_PRODUCT, (
    "Expected inner-product FAISS index"
)

print("Model:", EMBEDDING_MODEL)
print("Device:", next(embedder.parameters()).device)
print("Max sequence length:", embedder.max_seq_length)
print("Embedding dimensions:", embedding_dim)
print("FAISS vectors:", dense_index.ntotal)

print("\nLibrary versions:")
for package in (
    "sentence-transformers",
    "transformers",
    "torch",
    "faiss-cpu",
):
    print(f"{package}: {metadata.version(package)}")


# 3. Dense retrieval using existing index.
def search_dense(query, k=5):
    if not query.strip() or k <= 0:
        return []

    # E5 requires the query prefix.
    query_text = "query: " + query.strip()

    query_embedding = embedder.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    query_embedding = np.ascontiguousarray(
        query_embedding,
        dtype=np.float32,
    )

    assert query_embedding.shape == (1, dense_index.d)

    scores, indices = dense_index.search(
        query_embedding,
        min(k, dense_index.ntotal),
    )

    results = []

    for score, row_idx in zip(scores[0], indices[0]):
        if row_idx < 0:
            continue

        doc_id = dense_row_to_id[int(row_idx)]
        doc = doc_by_id[doc_id]

        results.append({
            "id": doc_id,
            "score": float(score),
            "label": doc["label"],
            "snippet": doc["text"][:350],
        })

    return results


# 4. Smoke test with the same BM25 query.
query = "πλημμύρες και αποζημιώσεις αγροτών"

dense_results = search_dense(query, k=5)

print("\nDense retrieval results:", len(dense_results))

for rank, result in enumerate(dense_results, start=1):
    print(
        f"\n#{rank} | {result['id']} "
        f"| score={result['score']:.4f}"
    )
    print("Category:", result["label"])
    print(result["snippet"])

assert len(dense_results) == 5

print("\nOK — dense retrieval working")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

/tmp/ipykernel_752/633508261.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = embedder.get_sentence_embedding_dimension()


Model: intfloat/multilingual-e5-small
Device: cpu
Max sequence length: 256
Embedding dimensions: 384
FAISS vectors: 5250

Library versions:
sentence-transformers: 5.7.0
transformers: 5.18.0
torch: 2.11.0+cpu
faiss-cpu: 1.15.1

Dense retrieval results: 5

#1 | macocu.el.8536036 | score=0.8748
Category: politics
Άρχισε η καταβολή από τον ΕΛΓΑ των προκαταβολών προς τους παραγωγούς βιομηχανικής ντομάτας Η βουλευτής Ηλείας του ΣΥΡΙΖΑ Έφη Γεωργοπούλου-Σαλτάρη ενημερώνει ότι ξεκίνησαν ήδη οι πρώτες προκαταβολές αποζημιώσεων για τις καταστροφές στην παραγωγή της βιομηχανικής ντομάτας του Νομού μας που συνέβησαν λόγω των πλημμυρών του Ιουνίου του 2018. Οι παραγωγ

#2 | macocu.el.6710186 | score=0.8732
Category: disaster, accident and emergency incident
Αφορά περιοχές τις χώρας που παρουσίασαν μείωση παραγωγής στο προανθικό στάδιο το 2021, λόγω των συνεχόμενων παγετών. Σύμφωνα με την ΚΥΑ, την οποία υπογράφουν ο υπουργός Αγροτικής Ανάπτυξης και Τροφίμων, Γιώργος Γεωργαντάς και ο αναπληρωτής υπουρ

In [9]:
# CELL 9 — Hybrid Retrieval with Reciprocal Rank Fusion

def search_hybrid(query, k=5, candidate_k=50, rrf_k=60):

    if not query.strip() or k <= 0:
        return []

    assert candidate_k > 0
    assert rrf_k > 0

    candidate_k = min(
        max(candidate_k, k),
        len(docs),
    )

    # 1. Retrieve candidates independently.
    bm25_results = search_bm25(
        query,
        k=candidate_k,
    )

    dense_results = search_dense(
        query,
        k=candidate_k,
    )

    # 2. Map document IDs to ranks.
    bm25_ranks = {
        result["id"]: rank
        for rank, result in enumerate(
            bm25_results, start=1
        )
    }

    dense_ranks = {
        result["id"]: rank
        for rank, result in enumerate(
            dense_results, start=1
        )
    }

    assert len(bm25_ranks) == len(bm25_results)
    assert len(dense_ranks) == len(dense_results)

    # 3. Union of candidates by document ID.
    candidate_ids = set(bm25_ranks) | set(dense_ranks)

    # 4. Reciprocal Rank Fusion.
    fused_results = []

    for doc_id in candidate_ids:

        bm25_rank = bm25_ranks.get(doc_id)
        dense_rank = dense_ranks.get(doc_id)

        rrf_score = 0.0

        if bm25_rank is not None:
            rrf_score += 1.0 / (rrf_k + bm25_rank)

        if dense_rank is not None:
            rrf_score += 1.0 / (rrf_k + dense_rank)

        doc = doc_by_id[doc_id]

        fused_results.append({
            "id": doc_id,
            "score": rrf_score,
            "bm25_rank": bm25_rank,
            "dense_rank": dense_rank,
            "label": doc["label"],
            "snippet": doc["text"][:350],
        })

    # 5. Deterministic sorting.
    fused_results.sort(
        key=lambda x: (
            -x["score"],
            min(
                rank for rank in (
                    x["bm25_rank"],
                    x["dense_rank"],
                )
                if rank is not None
            ),
            x["id"],
        )
    )

    return fused_results[:k]


# -------------------------------
# Functional smoke test
# -------------------------------

query = "πλημμύρες και αποζημιώσεις αγροτών"

hybrid_results = search_hybrid(
    query,
    k=5,
    candidate_k=50,
    rrf_k=60,
)

print("Query:", query)
print("Hybrid results:", len(hybrid_results))

for rank, result in enumerate(hybrid_results, start=1):

    print(f"\n#{rank} | {result['id']}")
    print(f"RRF score: {result['score']:.6f}")
    print(f"BM25 rank: {result['bm25_rank']}")
    print(f"Dense rank: {result['dense_rank']}")
    print("Category:", result["label"])
    print("Text:", result["snippet"][:200])


# -------------------------------
# Validation
# -------------------------------

assert len(hybrid_results) == 5

assert len({
    result["id"]
    for result in hybrid_results
}) == len(hybrid_results)

assert all(
    result["id"] in doc_by_id
    for result in hybrid_results
)

assert all(
    hybrid_results[i]["score"] >=
    hybrid_results[i + 1]["score"]
    for i in range(len(hybrid_results) - 1)
)

print("\nOK — hybrid retrieval working")

Query: πλημμύρες και αποζημιώσεις αγροτών
Hybrid results: 5

#1 | macocu.el.14374889
RRF score: 0.030579
BM25 rank: 3
Dense rank: 8
Category: disaster, accident and emergency incident
Text: Β. Νανόπουλος. Στηρίζουμε τα συμφέροντα των αγροτών μας 7 Σεπτεμβρίου 2022 Την φωνή αγωνίας και τα αιτήματα των παραγωγών του δήμου Κορινθίων που υπέστησαν πρόσφατα ζημιά στις καλλιέργειες τους, μετέφ

#2 | macocu.el.6710186
RRF score: 0.029828
BM25 rank: 13
Dense rank: 2
Category: disaster, accident and emergency incident
Text: Αφορά περιοχές τις χώρας που παρουσίασαν μείωση παραγωγής στο προανθικό στάδιο το 2021, λόγω των συνεχόμενων παγετών. Σύμφωνα με την ΚΥΑ, την οποία υπογράφουν ο υπουργός Αγροτικής Ανάπτυξης και Τροφίμ

#3 | macocu.el.4610325
RRF score: 0.026743
BM25 rank: 11
Dense rank: 19
Category: labour
Text: Ζεσταίνουν τις μηχανές οι αγρότες – Ζητούν προστασία των εισοδημάτων τους κόντρα στην ακρίβεια Όπως μετέδωσε, η Δημοσιογράφος της ΕΡΤ Ελπίδα Κουτσογιάννη από τον Τύρναβο, στις 11 το 

In [10]:
# CELL 10 — Frozen Retrieval Benchmark v2

import json
import random
import hashlib
from collections import Counter

SEED = 42
N_PER_TYPE = 300
DEV_PER_TYPE = 60

rng = random.Random(SEED)

# 1. Deterministic document ordering.
positions = list(range(len(docs)))
rng.shuffle(positions)

# 2. Find eligible late-query documents.
late_candidates = []

for idx in positions:
    words = docs[idx]["text"].split()

    if len(words) < 260:
        continue

    tail = " ".join(words[200:])

    sentences = re.split(
        r"(?<=[.!?;])\s+",
        tail,
    )

    valid = [
        s.strip()
        for s in sentences
        if len(s.split()) >= 12
    ]

    if valid:
        late_candidates.append((idx, valid))

print("Eligible late documents:", len(late_candidates))

assert len(late_candidates) >= N_PER_TYPE, (
    "Not enough late-query candidates"
)

# 3. Choose 300 late queries.
late_samples = []

for idx, sentences in late_candidates[:N_PER_TYPE]:
    query = rng.choice(sentences)
    late_samples.append((idx, query))

late_ids = {
    docs[idx]["id"]
    for idx, _ in late_samples
}

# 4. Choose 300 separate lead-query documents.
lead_samples = []

for idx in positions:
    doc = docs[idx]

    if doc["id"] in late_ids:
        continue

    words = doc["text"].split()

    if len(words) < 20:
        continue

    query = " ".join(words[:20])
    lead_samples.append((idx, query))

    if len(lead_samples) == N_PER_TYPE:
        break

assert len(lead_samples) == N_PER_TYPE

# 5. Shuffle each query type deterministically.
rng.shuffle(lead_samples)
rng.shuffle(late_samples)

# 6. Create development and evaluation records.
query_records = []

for query_type, samples in [
    ("lead", lead_samples),
    ("late", late_samples),
]:

    for number, (idx, query) in enumerate(
        samples, start=1
    ):
        partition = (
            "dev"
            if number <= DEV_PER_TYPE
            else "evaluation"
        )

        query_records.append({
            "query_id": f"{query_type}_{number:03d}",
            "query_type": query_type,
            "partition": partition,
            "source_id": docs[idx]["id"],
            "source_split": docs[idx]["split"],
            "query": query,
        })

# 7. Validate.
assert len(query_records) == 600

source_ids = [r["source_id"] for r in query_records]
assert len(source_ids) == len(set(source_ids))

dev_ids = {
    r["source_id"]
    for r in query_records
    if r["partition"] == "dev"
}

eval_ids = {
    r["source_id"]
    for r in query_records
    if r["partition"] == "evaluation"
}

assert dev_ids.isdisjoint(eval_ids)

counts = Counter(
    (r["partition"], r["query_type"])
    for r in query_records
)

print("\nQuery distribution:")

for key, count in sorted(counts.items()):
    print(key, count)

# 8. Persist the frozen query set.
OUTPUT_DIR = DATA_DIR / "results" / "retrieval_eval_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

queries_path = OUTPUT_DIR / "queries.jsonl"
manifest_path = OUTPUT_DIR / "manifest.json"

payload = "".join(
    json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n"
    for r in query_records
).encode("utf-8")

query_sha256 = hashlib.sha256(payload).hexdigest()

manifest = {
    "benchmark": "retrieval_eval_v2",
    "seed": SEED,
    "lead_queries": 300,
    "late_queries": 300,
    "dev_per_type": 60,
    "evaluation_per_type": 240,
    "query_generation": {
        "lead": "first 20 whitespace-separated words",
        "late": (
            "random sentence with >=12 words "
            "from text after word 200, "
            "articles with >=260 words"
        ),
    },
    "corpus": "EMMediaTopic Greek subset, 5250 articles",
    "index_repo": INDEX_REPO,
    "index_revision": index_revision,
    "queries_sha256": query_sha256,
    "source_file_sha256": {
        split: hashlib.sha256(
            paths[split].read_bytes()
        ).hexdigest()
        for split in ("train", "val", "test")
    },
}

manifest_bytes = (
    json.dumps(manifest, indent=2, ensure_ascii=False)
    + "\n"
).encode("utf-8")

# Prevent accidental modification of an existing benchmark.
for path, content in [
    (queries_path, payload),
    (manifest_path, manifest_bytes),
]:
    if path.exists():
        assert path.read_bytes() == content, (
            f"Existing benchmark differs: {path}"
        )
    else:
        path.write_bytes(content)

print("\nFiles:")
print(queries_path)
print(manifest_path)

print("\nQuery SHA256:", query_sha256)
print("Unique source articles:", len(set(source_ids)))
print("Dev/Evaluation overlap:", len(dev_ids & eval_ids))

assert len(dev_ids) == 120
assert len(eval_ids) == 480

print("\nOK — benchmark v2 frozen")

Eligible late documents: 1812

Query distribution:
('dev', 'late') 60
('dev', 'lead') 60
('evaluation', 'late') 240
('evaluation', 'lead') 240

Files:
/content/drive/MyDrive/greek-news-nlp/results/retrieval_eval_v2/queries.jsonl
/content/drive/MyDrive/greek-news-nlp/results/retrieval_eval_v2/manifest.json

Query SHA256: 987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccacac5
Unique source articles: 600
Dev/Evaluation overlap: 0

OK — benchmark v2 frozen


In [11]:
# CELL 11 — Matched-Text BM25 using E5 token budget

E5_PREFIX = "passage: "
MAX_E5_TOKENS = 256

tokenizer = embedder.tokenizer

assert tokenizer.is_fast, (
    "A fast tokenizer is required for offset mapping"
)

assert embedder.max_seq_length == MAX_E5_TOKENS

matched_texts = []
matched_char_lengths = []

BATCH_SIZE = 64

# 1. Apply the same E5 token budget.
for start_idx in range(0, len(docs), BATCH_SIZE):

    batch = docs[start_idx:start_idx + BATCH_SIZE]

    inputs = [
        E5_PREFIX + doc["text"]
        for doc in batch
    ]

    encoded = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_E5_TOKENS,
        add_special_tokens=True,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        padding=False,
    )

    for doc, offsets, special_mask in zip(
        batch,
        encoded["offset_mapping"],
        encoded["special_tokens_mask"],
    ):

        # Find the last covered character in the article.
        covered_end = max(
            (
                end - len(E5_PREFIX)
                for (_, end), is_special in zip(
                    offsets, special_mask
                )
                if not is_special and end > len(E5_PREFIX)
            ),
            default=0,
        )

        covered_end = max(
            0,
            min(covered_end, len(doc["text"]))
        )

        matched_text = doc["text"][:covered_end]

        matched_texts.append(matched_text)
        matched_char_lengths.append(covered_end)


# 2. Validate coverage.
assert len(matched_texts) == len(docs)

assert all(
    len(text) > 0
    for text in matched_texts
)

assert all(
    original["text"].startswith(matched)
    for original, matched in zip(docs, matched_texts)
)


# 3. Tokenize with the same BM25 preprocessing.
tokenized_matched_docs = [
    tokenize_greek(text)
    for text in matched_texts
]

assert all(
    len(tokens) > 0
    for tokens in tokenized_matched_docs
)


# 4. Build matched-text BM25 index.
bm25_matched = BM25Okapi(
    tokenized_matched_docs,
    k1=1.5,
    b=0.75,
    epsilon=0.25,
)


# 5. Search function.
def search_bm25_matched(query, k=5):

    if not query.strip() or k <= 0:
        return []

    tokens = tokenize_greek(query)

    if not tokens:
        return []

    scores = bm25_matched.get_scores(tokens)

    ranked_indices = np.argsort(
        -scores,
        kind="stable",
    )

    results = []

    for idx in ranked_indices:

        if scores[idx] <= 0:
            continue

        doc = docs[int(idx)]

        results.append({
            "id": doc["id"],
            "score": float(scores[idx]),
            "label": doc["label"],
            "snippet": matched_texts[int(idx)][:350],
        })

        if len(results) >= k:
            break

    return results


# 6. Smoke test.
query = "πλημμύρες και αποζημιώσεις αγροτών"

matched_results = search_bm25_matched(
    query,
    k=5,
)

truncated_count = sum(
    len(matched) < len(doc["text"])
    for matched, doc in zip(matched_texts, docs)
)

print("Total articles:", len(docs))
print("BM25 matched articles:", len(tokenized_matched_docs))
print("E5 token budget:", MAX_E5_TOKENS)
print("Articles shortened:", truncated_count)
print("Matched search results:", len(matched_results))

print("\nMatched BM25 results:")

for rank, result in enumerate(matched_results, start=1):
    print(
        f"#{rank} | {result['id']} "
        f"| score={result['score']:.4f}"
    )

assert len(matched_results) == 5

print("\nOK — matched-text BM25 working")

Total articles: 5250
BM25 matched articles: 5250
E5 token budget: 256
Articles shortened: 3614
Matched search results: 5

Matched BM25 results:
#1 | macocu.el.9211668 | score=15.3871
#2 | macocu.el.9799779 | score=13.9137
#3 | macocu.el.13132478 | score=13.2749
#4 | macocu.el.410901 | score=13.2049
#5 | macocu.el.3140995 | score=12.4573

OK — matched-text BM25 working


In [12]:
# CELL 12 — Frozen Queries and Text Coverage Audit

import json
import hashlib
from collections import Counter

# 1. Reload frozen queries.
saved_bytes = queries_path.read_bytes()

saved_queries = [
    json.loads(line)
    for line in saved_bytes.decode("utf-8").splitlines()
    if line.strip()
]

saved_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

# 2. Verify integrity.
assert hashlib.sha256(saved_bytes).hexdigest() == (
    saved_manifest["queries_sha256"]
)

assert len(saved_queries) == 600
assert len({q["query_id"] for q in saved_queries}) == 600
assert len({q["source_id"] for q in saved_queries}) == 600

# 3. Match source IDs to corpus positions.
id_to_position = {
    doc["id"]: i
    for i, doc in enumerate(docs)
}

assert len(id_to_position) == len(docs)

def normalize_whitespace(text):
    return " ".join(text.split()).casefold()

coverage = Counter()

# 4. Check literal query coverage.
for record in saved_queries:

    source_id = record["source_id"]
    query_type = record["query_type"]

    assert source_id in id_to_position

    idx = id_to_position[source_id]

    assert record["source_split"] == docs[idx]["split"]

    query_text = normalize_whitespace(
        record["query"]
    )

    full_text = normalize_whitespace(
        docs[idx]["text"]
    )

    matched_text = normalize_whitespace(
        matched_texts[idx]
    )

    assert query_text
    assert query_text in full_text

    coverage[(query_type, "total")] += 1

    if query_text in matched_text:
        coverage[(query_type, "inside_matched")] += 1
    else:
        coverage[(query_type, "outside_matched")] += 1


# 5. Report results.
print("Frozen query integrity: OK")
print("Query SHA256:", saved_manifest["queries_sha256"])

for query_type in ("lead", "late"):

    total = coverage[(query_type, "total")]

    inside = coverage[
        (query_type, "inside_matched")
    ]

    outside = coverage[
        (query_type, "outside_matched")
    ]

    assert inside + outside == total

    print(f"\n{query_type.upper()} QUERIES")
    print("Total:", total)
    print("Fully inside matched text:", inside)
    print("Not fully inside matched text:", outside)
    print(
        "Outside percentage:",
        f"{100 * outside / total:.2f}%"
    )


# 6. Corpus truncation statistics.
shortened = sum(
    len(matched) < len(doc["text"])
    for matched, doc in zip(matched_texts, docs)
)

print("\nCORPUS COVERAGE")
print("Total articles:", len(docs))
print("Shortened articles:", shortened)
print(
    "Shortened percentage:",
    f"{100 * shortened / len(docs):.2f}%"
)

assert coverage[("lead", "total")] == 300
assert coverage[("late", "total")] == 300

print("\nOK — retrieval coverage audit complete")

Frozen query integrity: OK
Query SHA256: 987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccacac5

LEAD QUERIES
Total: 300
Fully inside matched text: 300
Not fully inside matched text: 0
Outside percentage: 0.00%

LATE QUERIES
Total: 300
Fully inside matched text: 0
Not fully inside matched text: 300
Outside percentage: 100.00%

CORPUS COVERAGE
Total articles: 5250
Shortened articles: 3614
Shortened percentage: 68.84%

OK — retrieval coverage audit complete


In [14]:
# CELL 13 — Development Retrieval Evaluation

import numpy as np
from collections import Counter

# ------------------------------------
# 1. Development queries only
# ------------------------------------

dev_queries = [
    record
    for record in saved_queries
    if record["partition"] == "dev"
]

assert len(dev_queries) == 120

assert Counter(
    q["query_type"] for q in dev_queries
) == {"lead": 60, "late": 60}

CANDIDATE_K = 50
RRF_K = 60
TOP_K = 10

# ------------------------------------
# 2. Batch-encode dense queries on CPU
# ------------------------------------

query_inputs = [
    "query: " + record["query"].strip()
    for record in dev_queries
]

print("Encoding development queries...")

query_embeddings = embedder.encode(
    query_inputs,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

query_embeddings = np.ascontiguousarray(
    query_embeddings,
    dtype=np.float32,
)

assert query_embeddings.shape == (
    len(dev_queries),
    dense_index.d,
)

dense_scores, dense_indices = dense_index.search(
    query_embeddings,
    CANDIDATE_K,
)

print("Dense batch search complete.")

# ------------------------------------
# 3. RRF from existing ranked IDs
# ------------------------------------

def fuse_rankings(bm25_ids, dense_ids, k=10):

    bm25_ranks = {
        doc_id: rank
        for rank, doc_id in enumerate(
            bm25_ids, start=1
        )
    }

    dense_ranks = {
        doc_id: rank
        for rank, doc_id in enumerate(
            dense_ids, start=1
        )
    }

    candidate_ids = (
        set(bm25_ranks) | set(dense_ranks)
    )

    fused = []

    for doc_id in candidate_ids:

        b_rank = bm25_ranks.get(doc_id)
        d_rank = dense_ranks.get(doc_id)

        score = 0.0

        if b_rank is not None:
            score += 1.0 / (RRF_K + b_rank)

        if d_rank is not None:
            score += 1.0 / (RRF_K + d_rank)

        best_rank = min(
            rank
            for rank in (b_rank, d_rank)
            if rank is not None
        )

        fused.append(
            (doc_id, score, best_rank)
        )

    fused.sort(
        key=lambda x: (
            -x[1],
            x[2],
            x[0],
        )
    )

    return [
        item[0]
        for item in fused[:k]
    ]


# ------------------------------------
# 4. Evaluate all four retrievers
# ------------------------------------

evaluation_rows = []

for i, record in enumerate(dev_queries):

    query = record["query"]
    gold_id = record["source_id"]

    # BM25 Full
    bm25_full_ids = [
        r["id"]
        for r in search_bm25(
            query,
            k=CANDIDATE_K,
        )
    ]

    # BM25 Matched
    bm25_matched_ids = [
        r["id"]
        for r in search_bm25_matched(
            query,
            k=TOP_K,
        )
    ]

    # Dense E5 — reuse batch embeddings
    dense_ids = [
        dense_row_to_id[int(idx)]
        for idx in dense_indices[i]
        if idx >= 0
    ]

    # Hybrid RRF
    hybrid_ids = fuse_rankings(
        bm25_full_ids,
        dense_ids,
        k=TOP_K,
    )

    rankings = {
        "BM25 Full": bm25_full_ids[:TOP_K],
        "BM25 Matched": bm25_matched_ids,
        "Dense E5": dense_ids[:TOP_K],
        "Hybrid RRF": hybrid_ids,
    }

    for method, ranked_ids in rankings.items():

        assert len(ranked_ids) == len(set(ranked_ids))

        rank = next(
            (
                position
                for position, doc_id in enumerate(
                    ranked_ids, start=1
                )
                if doc_id == gold_id
            ),
            None,
        )

        evaluation_rows.append({
            "query_id": record["query_id"],
            "query_type": record["query_type"],
            "source_id": gold_id,
            "method": method,
            "rank": rank,
            "recall@1": int(
                rank is not None and rank <= 1
            ),
            "recall@5": int(
                rank is not None and rank <= 5
            ),
            "recall@10": int(
                rank is not None and rank <= 10
            ),
            "mrr@10": (
                1.0 / rank
                if rank is not None
                else 0.0
            ),
        })

    if (i + 1) % 20 == 0:
        print(
            f"Evaluated {i + 1}/{len(dev_queries)} queries"
        )


# ------------------------------------
# 5. Aggregate metrics
# ------------------------------------

import pandas as pd

dev_results = pd.DataFrame(evaluation_rows)

assert len(dev_results) == 120 * 4

methods = [
    "BM25 Full",
    "BM25 Matched",
    "Dense E5",
    "Hybrid RRF",
]

metric_columns = [
    "recall@1",
    "recall@5",
    "recall@10",
    "mrr@10",
]

for query_type in ["lead", "late", "all"]:

    if query_type == "all":
        subset = dev_results
    else:
        subset = dev_results[
            dev_results["query_type"] == query_type
        ]

    summary = (
        subset.groupby("method")[metric_columns]
        .mean()
        .reindex(methods)
    )

    assert not summary.isna().any().any()

    print("\n" + "=" * 55)
    print(f"DEVELOPMENT RESULTS — {query_type.upper()}")
    print("=" * 55)

    print(summary.round(4).to_string())


# ------------------------------------
# 6. Final checks
# ------------------------------------

assert set(
    dev_results["query_id"]
) == {
    q["query_id"]
    for q in dev_queries
}

assert all(
    dev_results[col].between(0, 1).all()
    for col in metric_columns
)

assert all(
    dev_results["recall@1"]
    <= dev_results["recall@5"]
)

assert all(
    dev_results["recall@5"]
    <= dev_results["recall@10"]
)

print("\nOK — development evaluation complete")

Encoding development queries...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Dense batch search complete.
Evaluated 20/120 queries
Evaluated 40/120 queries
Evaluated 60/120 queries
Evaluated 80/120 queries
Evaluated 100/120 queries
Evaluated 120/120 queries

DEVELOPMENT RESULTS — LEAD
              recall@1  recall@5  recall@10  mrr@10
method                                             
BM25 Full          1.0       1.0        1.0     1.0
BM25 Matched       1.0       1.0        1.0     1.0
Dense E5           1.0       1.0        1.0     1.0
Hybrid RRF         1.0       1.0        1.0     1.0

DEVELOPMENT RESULTS — LATE
              recall@1  recall@5  recall@10  mrr@10
method                                             
BM25 Full       1.0000      1.00     1.0000  1.0000
BM25 Matched    0.3500      0.45     0.4833  0.3943
Dense E5        0.3833      0.50     0.6000  0.4438
Hybrid RRF      0.6000      0.85     0.9833  0.7120

DEVELOPMENT RESULTS — ALL
              recall@1  recall@5  recall@10  mrr@10
method                                             
BM25 Ful

In [15]:
# CELL 14 — Development Failure Analysis

# 1. Compare late-query rankings.
late_dev = dev_results[
    dev_results["query_type"] == "late"
]

rank_table = late_dev.pivot(
    index="query_id",
    columns="method",
    values="rank",
).reindex(columns=methods)

print("LATE DEVELOPMENT — TOP-10 HITS")

for method in methods:
    hits = rank_table[method].notna().sum()
    print(f"{method}: {hits}/60")

# 2. Identify Hybrid failures.
hybrid_misses = rank_table[
    rank_table["Hybrid RRF"].isna()
]

print("\nHybrid Top-10 failures:", len(hybrid_misses))
print(hybrid_misses.to_string())

# 3. Investigate each missed query.
query_lookup = {
    q["query_id"]: (i, q)
    for i, q in enumerate(dev_queries)
}

for query_id in hybrid_misses.index:

    i, record = query_lookup[query_id]

    query = record["query"]
    gold_id = record["source_id"]

    bm25_results = search_bm25(
        query,
        k=CANDIDATE_K,
    )

    b_ids = [r["id"] for r in bm25_results]

    d_ids = [
        dense_row_to_id[int(idx)]
        for idx in dense_indices[i]
        if idx >= 0
    ]

    all_candidates = set(b_ids) | set(d_ids)

    full_hybrid = fuse_rankings(
        b_ids,
        d_ids,
        k=len(all_candidates),
    )

    def get_rank(ids, target):
        try:
            return ids.index(target) + 1
        except ValueError:
            return None

    print("\n" + "=" * 55)
    print("Query ID:", query_id)
    print("Gold article:", gold_id)
    print("Query:", query[:400])

    print("\nCandidate rankings:")
    print("BM25 Full:", get_rank(b_ids, gold_id))
    print("Dense E5:", get_rank(d_ids, gold_id))
    print("Hybrid RRF:", get_rank(full_hybrid, gold_id))

    print("\nHybrid Top-10 IDs:")
    for rank, doc_id in enumerate(full_hybrid[:10], 1):
        print(f"{rank}. {doc_id}")

print("\nOK — development failure analysis complete")

LATE DEVELOPMENT — TOP-10 HITS
BM25 Full: 60/60
BM25 Matched: 29/60
Dense E5: 36/60
Hybrid RRF: 59/60

Hybrid Top-10 failures: 1
method    BM25 Full  BM25 Matched  Dense E5  Hybrid RRF
query_id                                               
late_044        1.0           NaN       NaN         NaN

Query ID: late_044
Gold article: macocu.el.387380
Query: * Ο Σαΐντ Πρίτζετ είναι πρώτος, μεταξύ των παικτών της ομάδας του, σε πόντους (159), κλεψίματα (19) και ranking (150).

Candidate rankings:
BM25 Full: 1
Dense E5: None
Hybrid RRF: 11

Hybrid Top-10 IDs:
1. macocu.el.12121653
2. macocu.el.5489244
3. macocu.el.136779
4. macocu.el.7455310
5. macocu.el.6907596
6. macocu.el.14380719
7. macocu.el.6900292
8. macocu.el.11187151
9. macocu.el.12254697
10. macocu.el.10818164

OK — development failure analysis complete


In [20]:
# CELL 14A — Diagnose SHA256 mismatch

import hashlib
import json

actual_bytes = queries_path.read_bytes()

actual_sha = hashlib.sha256(
    actual_bytes
).hexdigest()

manifest_disk = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

expected_sha = (
    "987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccac5"
)

print("Queries file:", queries_path)
print()

print("EXPECTED SHA:")
print(expected_sha)

print("\nACTUAL SHA:")
print(actual_sha)

print("\nMANIFEST SHA:")
print(manifest_disk["queries_sha256"])

print("\nIN-MEMORY SHA:")
print(hashlib.sha256(saved_bytes).hexdigest())

print("\nCHECKS:")
print("Actual == Expected:", actual_sha == expected_sha)
print(
    "Actual == Manifest:",
    actual_sha == manifest_disk["queries_sha256"]
)
print(
    "Actual == In-memory:",
    actual_bytes == saved_bytes
)

records = [
    json.loads(line)
    for line in actual_bytes.decode("utf-8").splitlines()
    if line.strip()
]

print("\nQuery count:", len(records))
print(
    "Unique query IDs:",
    len(set(r["query_id"] for r in records))
)

print("\nDiagnostic complete.")

Queries file: /content/drive/MyDrive/greek-news-nlp/results/retrieval_eval_v2/queries.jsonl

EXPECTED SHA:
987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccac5

ACTUAL SHA:
987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccacac5

MANIFEST SHA:
987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccacac5

IN-MEMORY SHA:
987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccacac5

CHECKS:
Actual == Expected: False
Actual == Manifest: True
Actual == In-memory: True

Query count: 600
Unique query IDs: 600

Diagnostic complete.


In [21]:
# CELL 15 — Frozen Final Retrieval Evaluation (Fixed)

import pandas as pd
import numpy as np
import hashlib
from collections import Counter

# ------------------------------------
# 1. Verify frozen benchmark
# ------------------------------------

EXPECTED_QUERY_SHA = (
    "987f805360d3a6485b111a65f15f8fb2d159389445c8d1640bc67a786ccacac5"
)

assert hashlib.sha256(
    queries_path.read_bytes()
).hexdigest() == EXPECTED_QUERY_SHA

assert saved_manifest["queries_sha256"] == EXPECTED_QUERY_SHA

eval_queries = [
    q for q in saved_queries
    if q["partition"] == "evaluation"
]

assert len(eval_queries) == 480

assert Counter(
    q["query_type"] for q in eval_queries
) == {"lead": 240, "late": 240}

assert {
    q["source_id"] for q in eval_queries
}.isdisjoint({
    q["source_id"] for q in dev_queries
})

# Freeze evaluation parameters
assert CANDIDATE_K == 50
assert RRF_K == 60
assert TOP_K == 10

print("Frozen benchmark verified.")
print("Evaluation queries:", len(eval_queries))

# ------------------------------------
# 2. Batch encode on CPU
# ------------------------------------

eval_inputs = [
    "query: " + q["query"].strip()
    for q in eval_queries
]

print("\nEncoding evaluation queries...")

eval_embeddings = embedder.encode(
    eval_inputs,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

eval_embeddings = np.ascontiguousarray(
    eval_embeddings,
    dtype=np.float32
)

assert eval_embeddings.shape == (
    480,
    dense_index.d
)

eval_dense_scores, eval_dense_indices = (
    dense_index.search(
        eval_embeddings,
        CANDIDATE_K
    )
)

print("Dense retrieval completed.")

# ------------------------------------
# 3. Evaluate four retrieval methods
# ------------------------------------

final_eval_rows = []

for i, record in enumerate(eval_queries):

    query = record["query"]
    gold_id = record["source_id"]

    # BM25 Full
    bm25_full_ids = [
        r["id"]
        for r in search_bm25(
            query,
            k=CANDIDATE_K
        )
    ]

    # BM25 Matched
    bm25_matched_ids = [
        r["id"]
        for r in search_bm25_matched(
            query,
            k=TOP_K
        )
    ]

    # Dense E5
    dense_ranked_ids = [
        dense_row_to_id[int(idx)]
        for idx in eval_dense_indices[i]
        if idx >= 0
    ]

    # Hybrid RRF
    hybrid_ids = fuse_rankings(
        bm25_full_ids,
        dense_ranked_ids,
        k=TOP_K
    )

    rankings = {
        "BM25 Full": bm25_full_ids[:TOP_K],
        "BM25 Matched": bm25_matched_ids,
        "Dense E5": dense_ranked_ids[:TOP_K],
        "Hybrid RRF": hybrid_ids,
    }

    for method, ranked_ids in rankings.items():

        assert len(ranked_ids) == len(set(ranked_ids))

        rank = next(
            (
                position
                for position, doc_id in enumerate(
                    ranked_ids, start=1
                )
                if doc_id == gold_id
            ),
            None
        )

        final_eval_rows.append({
            "query_id": record["query_id"],
            "query_type": record["query_type"],
            "source_id": gold_id,
            "method": method,
            "rank": rank,
            "recall@1": int(
                rank is not None and rank <= 1
            ),
            "recall@5": int(
                rank is not None and rank <= 5
            ),
            "recall@10": int(
                rank is not None and rank <= 10
            ),
            "mrr@10": (
                1.0 / rank
                if rank is not None
                else 0.0
            ),
        })

    if (i + 1) % 80 == 0:
        print(
            f"Evaluated {i + 1}/{len(eval_queries)} queries"
        )

# ------------------------------------
# 4. Validate results
# ------------------------------------

final_eval_results = pd.DataFrame(final_eval_rows)

assert len(final_eval_results) == 480 * 4

assert final_eval_results.groupby(
    "query_id"
).size().eq(4).all()

assert all(
    final_eval_results[col].between(0, 1).all()
    for col in metric_columns
)

assert all(
    final_eval_results["recall@1"]
    <= final_eval_results["recall@5"]
)

assert all(
    final_eval_results["recall@5"]
    <= final_eval_results["recall@10"]
)

# ------------------------------------
# 5. Report final metrics
# ------------------------------------

for query_type in ("lead", "late", "all"):

    if query_type == "all":
        subset = final_eval_results
    else:
        subset = final_eval_results[
            final_eval_results["query_type"]
            == query_type
        ]

    summary = (
        subset.groupby("method")[metric_columns]
        .mean()
        .reindex(methods)
    )

    assert not summary.isna().any().any()

    print("\n" + "=" * 55)
    print(f"FINAL EVALUATION — {query_type.upper()}")
    print("=" * 55)

    print(summary.round(4).to_string())

print("\nOK — final retrieval evaluation complete")
print("Results are in memory; not saved yet.")

Frozen benchmark verified.
Evaluation queries: 480

Encoding evaluation queries...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Dense retrieval completed.
Evaluated 80/480 queries
Evaluated 160/480 queries
Evaluated 240/480 queries
Evaluated 320/480 queries
Evaluated 400/480 queries
Evaluated 480/480 queries

FINAL EVALUATION — LEAD
              recall@1  recall@5  recall@10  mrr@10
method                                             
BM25 Full       1.0000       1.0        1.0  1.0000
BM25 Matched    1.0000       1.0        1.0  1.0000
Dense E5        0.9958       1.0        1.0  0.9979
Hybrid RRF      1.0000       1.0        1.0  1.0000

FINAL EVALUATION — LATE
              recall@1  recall@5  recall@10  mrr@10
method                                             
BM25 Full       1.0000    1.0000     1.0000  1.0000
BM25 Matched    0.2542    0.4042     0.4583  0.3224
Dense E5        0.3125    0.4542     0.5125  0.3714
Hybrid RRF      0.4958    0.8250     0.9583  0.6363

FINAL EVALUATION — ALL
              recall@1  recall@5  recall@10  mrr@10
method                                             
BM25 Full       

In [22]:
# CELL 16 — Save Final Results and Reproducibility Manifest

import json
import hashlib
import sys
import importlib.metadata as metadata
from pathlib import Path

# ------------------------------------
# 1. Verify benchmark integrity
# ------------------------------------

RESULTS_DIR = DATA_DIR / "results" / "retrieval_eval_v2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert hashlib.sha256(
    queries_path.read_bytes()
).hexdigest() == saved_manifest["queries_sha256"]

for split, expected_hash in saved_manifest[
    "source_file_sha256"
].items():

    actual_hash = hashlib.sha256(
        paths[split].read_bytes()
    ).hexdigest()

    assert actual_hash == expected_hash, (
        f"Dataset integrity mismatch: {split}"
    )

assert len(final_eval_results) == 1920

assert final_eval_results.groupby(
    ["query_id", "method"]
).size().eq(1).all()

print("Benchmark integrity: OK")

# ------------------------------------
# 2. Prepare per-query results
# ------------------------------------

result_columns = [
    "query_id",
    "query_type",
    "source_id",
    "method",
    "rank",
    "recall@1",
    "recall@5",
    "recall@10",
    "mrr@10",
]

results_export = final_eval_results[
    result_columns
].copy()

results_export["rank"] = (
    results_export["rank"].astype("Int64")
)

results_export = results_export.sort_values(
    ["query_type", "query_id", "method"]
).reset_index(drop=True)

predictions_bytes = results_export.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.12g",
).encode("utf-8")

# ------------------------------------
# 3. Prepare aggregate metrics
# ------------------------------------

summary_rows = []

for query_type in ("lead", "late", "all"):

    subset = (
        final_eval_results
        if query_type == "all"
        else final_eval_results[
            final_eval_results["query_type"] == query_type
        ]
    )

    expected_count = (
        480 if query_type == "all" else 240
    )

    for method in methods:

        method_rows = subset[
            subset["method"] == method
        ]

        assert len(method_rows) == expected_count

        summary_rows.append({
            "query_type": query_type,
            "method": method,
            "n_queries": len(method_rows),
            **{
                metric: float(method_rows[metric].mean())
                for metric in metric_columns
            },
        })

final_metrics = pd.DataFrame(summary_rows)

assert len(final_metrics) == 12

metrics_bytes = final_metrics.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.10f",
).encode("utf-8")

# ------------------------------------
# 4. Record index and model provenance
# ------------------------------------

index_files = {
    name: INDEX_DIR / name
    for name in ("articles.faiss", "docs.json")
}

index_hashes = {}

for name, path in index_files.items():

    assert path.is_file(), f"Missing index file: {path}"

    index_hashes[name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

# Use the revision reported by the loaded model
# configuration if available.
model_config = getattr(
    getattr(embedder._first_module(), "auto_model", None),
    "config",
    None,
)

model_revision = getattr(
    model_config,
    "_commit_hash",
    None,
)

software_versions = {
    "python": sys.version.split()[0],
}

for package in (
    "numpy",
    "pandas",
    "torch",
    "sentence-transformers",
    "transformers",
    "faiss-cpu",
    "rank-bm25",
    "huggingface-hub",
):
    software_versions[package] = metadata.version(package)

# ------------------------------------
# 5. Create reproducibility manifest
# ------------------------------------

evaluation_manifest = {
    "benchmark": "retrieval_eval_v2",
    "phase": "final_evaluation",

    "dataset": {
        "corpus_size": len(docs),
        "source_file_sha256": saved_manifest[
            "source_file_sha256"
        ],
        "queries_sha256": saved_manifest[
            "queries_sha256"
        ],
        "evaluation_queries": 480,
        "lead_queries": 240,
        "late_queries": 240,
        "gold_definition": "source article document_id",
    },

    "dense_retrieval": {
        "model": EMBEDDING_MODEL,
        "loaded_model_revision_if_available": model_revision,
        "device": "cpu",
        "max_seq_length": 256,
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
        "normalize_embeddings": True,
        "embedding_dimensions": dense_index.d,
        "faiss_metric": "inner_product",
        "index_repo": INDEX_REPO,
        "index_revision": index_revision,
        "index_sha256": index_hashes,
    },

    "bm25": {
        "k1": 1.5,
        "b": 0.75,
        "epsilon": 0.25,
        "variants": [
            "full_text",
            "matched_e5_token_budget",
        ],
    },

    "hybrid": {
        "method": "reciprocal_rank_fusion",
        "candidate_k": CANDIDATE_K,
        "rrf_k": RRF_K,
        "top_k": TOP_K,
        "inputs": ["BM25 Full", "Dense E5"],
        "weights": "equal",
    },

    "metrics": metric_columns,

    "software_versions": software_versions,

    "limitations": [
        "Queries are extracted verbatim from source articles.",
        "Gold relevance is defined by source document ID.",
        "All source documents are included in the search index.",
        "This is not an evaluation on human-written questions.",
        "Matched BM25 shares an E5-derived text budget, "
        "not the E5 tokenization or embedding representation.",
        "The loaded E5 model revision is recorded only "
        "if available from its configuration.",
    ],

    "result_files_sha256": {
        "evaluation_predictions.csv": hashlib.sha256(
            predictions_bytes
        ).hexdigest(),
        "evaluation_metrics.csv": hashlib.sha256(
            metrics_bytes
        ).hexdigest(),
    },
}

manifest_bytes = (
    json.dumps(
        evaluation_manifest,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")

# ------------------------------------
# 6. Save files safely
# ------------------------------------

output_files = {
    "evaluation_predictions.csv": predictions_bytes,
    "evaluation_metrics.csv": metrics_bytes,
    "evaluation_manifest.json": manifest_bytes,
}

for filename, content in output_files.items():

    output_path = RESULTS_DIR / filename

    # Prevent accidental overwriting.
    if output_path.exists():

        assert output_path.read_bytes() == content, (
            f"Existing file differs: {output_path}"
        )

        print("Verified existing:", filename)

    else:

        output_path.write_bytes(content)

        print("Saved:", filename)

    assert hashlib.sha256(
        output_path.read_bytes()
    ).digest() == hashlib.sha256(content).digest()

# ------------------------------------
# 7. Final verification
# ------------------------------------

print("\nOutput directory:")
print(RESULTS_DIR)

print("\nSaved evaluation rows:", len(results_export))
print("Saved summary rows:", len(final_metrics))

print("\nFinal metrics:")
print(
    final_metrics.round(4).to_string(index=False)
)

print("\nOK — final results and manifest saved")

Benchmark integrity: OK
Saved: evaluation_predictions.csv
Saved: evaluation_metrics.csv
Saved: evaluation_manifest.json

Output directory:
/content/drive/MyDrive/greek-news-nlp/results/retrieval_eval_v2

Saved evaluation rows: 1920
Saved summary rows: 12

Final metrics:
query_type       method  n_queries  recall@1  recall@5  recall@10  mrr@10
      lead    BM25 Full        240    1.0000    1.0000     1.0000  1.0000
      lead BM25 Matched        240    1.0000    1.0000     1.0000  1.0000
      lead     Dense E5        240    0.9958    1.0000     1.0000  0.9979
      lead   Hybrid RRF        240    1.0000    1.0000     1.0000  1.0000
      late    BM25 Full        240    1.0000    1.0000     1.0000  1.0000
      late BM25 Matched        240    0.2542    0.4042     0.4583  0.3224
      late     Dense E5        240    0.3125    0.4542     0.5125  0.3714
      late   Hybrid RRF        240    0.4958    0.8250     0.9583  0.6363
       all    BM25 Full        480    1.0000    1.0000     1.00

In [23]:
# CELL 17 — Verify Saved Evaluation Results

import json
import hashlib
import numpy as np
import pandas as pd

RESULTS_DIR = DATA_DIR / "results" / "retrieval_eval_v2"

# 1. Load saved files
pred_path = RESULTS_DIR / "evaluation_predictions.csv"
metrics_path = RESULTS_DIR / "evaluation_metrics.csv"
manifest_path_final = RESULTS_DIR / "evaluation_manifest.json"

pred = pd.read_csv(pred_path)
metrics = pd.read_csv(metrics_path)

with open(manifest_path_final, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("Files loaded successfully.")

# 2. Verify SHA256 checksums
for filename in (
    "evaluation_predictions.csv",
    "evaluation_metrics.csv",
):
    file_path = RESULTS_DIR / filename

    actual_sha = hashlib.sha256(
        file_path.read_bytes()
    ).hexdigest()

    expected_sha = manifest[
        "result_files_sha256"
    ][filename]

    assert actual_sha == expected_sha

print("SHA256 verification: OK")

# 3. Verify number of rows and unique queries
assert len(pred) == 1920
assert len(metrics) == 12
assert pred["query_id"].nunique() == 480

assert pred.groupby("query_id").size().eq(4).all()

assert not pred.duplicated(
    ["query_id", "method"]
).any()

assert not pred["source_id"].isna().any()

assert pred["rank"].dropna().between(1, 10).all()

print("Dataset structure: OK")

# 4. Verify stored query identity
frozen_eval = {
    q["query_id"]: (
        q["source_id"],
        q["query_type"]
    )
    for q in saved_queries
    if q["partition"] == "evaluation"
}

assert len(frozen_eval) == 480

for record in pred.itertuples():

    assert record.query_id in frozen_eval

    expected_source, expected_type = frozen_eval[
        record.query_id
    ]

    assert record.source_id == expected_source
    assert record.query_type == expected_type

print("Frozen query identities: OK")

# 5. Recalculate metrics from saved predictions
metric_columns = [
    "recall@1",
    "recall@5",
    "recall@10",
    "mrr@10",
]

for query_type in ("lead", "late", "all"):

    subset = (
        pred
        if query_type == "all"
        else pred[pred["query_type"] == query_type]
    )

    for method in methods:

        rows = subset[subset["method"] == method]

        stored = metrics[
            (metrics["query_type"] == query_type)
            & (metrics["method"] == method)
        ]

        assert len(stored) == 1

        assert len(rows) == int(
            stored.iloc[0]["n_queries"]
        )

        for metric in metric_columns:

            calculated = rows[metric].mean()
            saved = stored.iloc[0][metric]

            assert np.isclose(
                calculated,
                saved,
                atol=1e-9,
                rtol=0,
            ), (
                f"Mismatch: {query_type}, "
                f"{method}, {metric}"
            )

print("Metric recalculation: OK")

# 6. Final summary
print("\nSaved files:")
for file_path in sorted(RESULTS_DIR.iterdir()):
    if file_path.is_file():
        print("-", file_path.name)

print("\nEvaluation summary:")
print(metrics.round(4).to_string(index=False))

print("\nOK — Notebook 08 results verified")

Files loaded successfully.
SHA256 verification: OK
Dataset structure: OK
Frozen query identities: OK
Metric recalculation: OK

Saved files:
- evaluation_manifest.json
- evaluation_metrics.csv
- evaluation_predictions.csv
- manifest.json
- queries.jsonl

Evaluation summary:
query_type       method  n_queries  recall@1  recall@5  recall@10  mrr@10
      lead    BM25 Full        240    1.0000    1.0000     1.0000  1.0000
      lead BM25 Matched        240    1.0000    1.0000     1.0000  1.0000
      lead     Dense E5        240    0.9958    1.0000     1.0000  0.9979
      lead   Hybrid RRF        240    1.0000    1.0000     1.0000  1.0000
      late    BM25 Full        240    1.0000    1.0000     1.0000  1.0000
      late BM25 Matched        240    0.2542    0.4042     0.4583  0.3224
      late     Dense E5        240    0.3125    0.4542     0.5125  0.3714
      late   Hybrid RRF        240    0.4958    0.8250     0.9583  0.6363
       all    BM25 Full        480    1.0000    1.0000     1

## Final Results — Hybrid Retrieval Benchmark v2

### Objective

Evaluate lexical, dense, and hybrid retrieval approaches
on a corpus of 5,250 Greek news articles.

Four retrieval methods were compared:

1. **BM25 Full:** BM25 over complete article texts.
2. **BM25 Matched:** BM25 over article prefixes limited
   using the E5 tokenizer's 256-token input budget.
3. **Dense E5:** `intfloat/multilingual-e5-small` with
   normalized embeddings and FAISS inner-product search.
4. **Hybrid RRF:** Reciprocal Rank Fusion combining
   BM25 Full and Dense E5.

### Evaluation Protocol

- Corpus: 5,250 Greek news articles.
- Query generation seed: 42.
- Total queries: 600.
- Development: 120 queries (60 lead, 60 late).
- Final evaluation: 480 queries (240 lead, 240 late).
- No overlapping source article IDs between partitions.
- Gold document: original article from which the query
  was extracted.
- Metrics: Recall@1, Recall@5, Recall@10, MRR@10.

**Hybrid configuration:**
- Candidate pool: 50 documents per retriever.
- RRF constant: 60.
- Equal contributions from BM25 Full and Dense E5.
- Maximum reported retrieval depth: 10.

Parameters were kept unchanged for final evaluation.

### Final Evaluation Results — Late Queries

| Method | Recall@1 | Recall@5 | Recall@10 | MRR@10 |
|---|---:|---:|---:|---:|
| BM25 Full | 1.0000 | 1.0000 | 1.0000 | 1.0000 |
| BM25 Matched | 0.2542 | 0.4042 | 0.4583 | 0.3224 |
| Dense E5 | 0.3125 | 0.4542 | 0.5125 | 0.3714 |
| Hybrid RRF | 0.4958 | 0.8250 | 0.9583 | 0.6363 |

### Observations

- BM25 Full achieved perfect source-document retrieval
  on this benchmark, where queries were extracted
  verbatim from indexed articles.
- Hybrid RRF improved late-query Recall@10 from
  0.5125 (Dense E5) to 0.9583.
- BM25 Matched achieved 0.4583 Recall@10, showing
  the impact of restricted article text coverage.
- Hybrid RRF did not outperform BM25 Full.
- All four methods achieved Recall@10 = 1.0 on
  lead queries.

### Text Coverage Audit

- 3,614 out of 5,250 articles (68.84%) were shortened
  under the E5 token-budget comparison.
- All 300 lead queries were fully contained in the
  matched article text.
- None of the 300 late queries were fully contained
  in the matched article text.

### Limitations

1. Queries were extracted directly from the indexed
   documents, rather than independently written by users.
2. Source articles were included in the retrieval index.
3. The evaluation measures source-document retrieval,
   not general relevance to natural-language questions.
4. Full-text BM25 has an advantage when retrieving
   verbatim passages from articles.
5. BM25 Matched shares an E5-derived text coverage
   limit but not the same tokenization or representation.
6. A late query not fully contained in the E5 prefix
   may still overlap partially with that prefix.
7. No reranker was evaluated in this experiment.
8. These experiments do not establish comparative
   production latency or GPU memory requirements.

### Conclusion

Hybrid RRF substantially improved retrieval over
Dense E5 on late source-document queries.

However, BM25 Full remained superior in this
benchmark, highlighting the importance of text
coverage and the limitations of extractive query
generation.

Future evaluation should include independently
written, realistic questions and human-assessed
relevance judgments before making broader claims
about retrieval effectiveness.

### Reproducibility

Saved artifacts:

- `queries.jsonl`
- `manifest.json`
- `evaluation_predictions.csv`
- `evaluation_metrics.csv`
- `evaluation_manifest.json`

All artifacts are stored under:

`results/retrieval_eval_v2/`

The evaluation manifest records configuration,
software versions, source hashes, index provenance,
and benchmark limitations.